In [1]:
%load_ext autoreload
%autoreload 2

import logging
from fabduckdb import register_function
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime

logging.basicConfig(
    format="%(asctime)s %(name)s %(levelname)-8s %(message)s",
    level=logging.DEBUG,
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger("fab_duckdb")
logger.setLevel(logging.DEBUG)

In [2]:
def df_creator(x: int) -> pd.DataFrame:
    start_date = datetime(2023, 1, 1)
    end_date = datetime(2023, 12, 31)
    num_rows = x

    datetime_col = pd.date_range(start_date, end_date, periods=num_rows)

    # Generate random numeric values
    numeric_col1 = np.random.rand(num_rows)
    numeric_col2 = np.random.randint(1, 100, num_rows)

    # Generate random string values
    string_col1 = np.random.choice(["apple", "banana", "cherry"], num_rows)
    string_col2 = np.random.choice(["red", "green", "blue"], num_rows)

    # Create the pandas DataFrame
    data = {
        "DateTime": datetime_col,
        "Numeric1": numeric_col1,
        "Numeric2": numeric_col2,
        "String1": string_col1,
        "String2": string_col2,
    }

    df = pd.DataFrame(data)
    return df


display(df_creator(2))

,DateTime,Numeric1,Numeric2,String1,String2
0,2023-01-01,0.407163,11,cherry,green
1,2023-12-31,0.823505,66,apple,green


In [3]:
"""Example using a function"""
from fabduckdb.table_functions import fab_functions

fab_functions.register_function("mydfcreator", df_creator, generates_filepath=False)

In [4]:
"""Example using a lambda & DataFrame"""
register_function(
    "dfcreate",
    lambda rows, cols, con=None: pd.DataFrame(np.random.rand(rows, cols)),
    generates_filepath=False,
)

duckdb.connect().execute("select * from dfcreate(cols=4,rows=3)").df()

2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    # Statements: 1, ['select * from dfcreate(cols=4,rows=3)']
2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    Processing: select * from dfcreate(cols=4,rows=3)
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Extracted subquery: dfcreate.None at 23:36
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Replacing dfcreate(cols=4,rows=3) with __fabduck_1
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    select * from dfcreate(cols=4,rows=3) rewritten to select * from __fabduck_1, {'__fabduck_1': ContextObject(functionname='dfcreate', functioncall='dfcreate(cols=4,rows=3)', params='cols=4,rows=3', data=None, name='__fabduck_1', is_file=False)}
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Executing subquery dfcreate(cols=4,rows=3)
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Running dfcreate against cols=4,rows=3
2025-08-28 18:31:28 fabd

,0,1,2,3
0,0.048534,0.831191,0.359109,0.898223
1,0.910908,0.205601,0.097877,0.902172
2,0.303668,0.005639,0.121819,0.520795


In [5]:
"""Example returning a filepath to a parquet file"""

register_function(
    "dfcreate",
    lambda rows, cols, filename, con: pd.DataFrame(
        np.random.rand(rows, cols)
    ).to_parquet(filename),
    generates_filepath=True,
)

duckdb.connect().execute("select * from dfcreate(3,4)").df()

2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    # Statements: 1, ['select * from dfcreate(3,4)']
2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    Processing: select * from dfcreate(3,4)
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Extracted subquery: dfcreate.None at 23:26
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Replacing dfcreate(3,4) with __fabduck_1.parquet
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    select * from dfcreate(3,4) rewritten to select * from __fabduck_1.parquet, {'__fabduck_1.parquet': ContextObject(functionname='dfcreate', functioncall='dfcreate(3,4)', params="3,4,filename='__fabduck_1.parquet'", data=None, name='__fabduck_1', is_file=True)}
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Executing subquery dfcreate(3,4)
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Running dfcreate against 3,4,filename='__fabduck_1.parquet'
2025-08-28 18:31:28

,0,1,2,3
0,0.507702,0.412383,0.562931,0.111059
1,0.548278,0.644956,0.819772,0.582668
2,0.300711,0.679205,0.193001,0.307958


In [6]:
"""Example returning a pyarrow table"""

register_function(
    "dfcreate_pa",
    lambda rows, cols, con=None: __import__("pyarrow").table(
        {
            f"col{i}": [
                "".join(
                    __import__("random").choices(
                        __import__("string").ascii_letters, k=5
                    )
                )
                for _ in range(rows)
            ]
            for i in range(cols)
        }
    ),
    generates_filepath=False,
)

duckdb.connect().execute("select * from dfcreate_pa(3,4)").df()

2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    # Statements: 1, ['select * from dfcreate_pa(3,4)']
2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    Processing: select * from dfcreate_pa(3,4)
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Extracted subquery: dfcreate_pa.None at 26:29
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Replacing dfcreate_pa(3,4) with __fabduck_1
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    select * from dfcreate_pa(3,4) rewritten to select * from __fabduck_1, {'__fabduck_1': ContextObject(functionname='dfcreate_pa', functioncall='dfcreate_pa(3,4)', params='3,4', data=None, name='__fabduck_1', is_file=False)}
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Executing subquery dfcreate_pa(3,4)
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Running dfcreate_pa against 3,4
2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    Registering __fabduck_1
20

,col0,col1,col2,col3
0,TXnmX,nWFcg,HPXhW,djCOc
1,NZNih,PpFQf,GkHPF,DWlhe
2,kMbXv,inxAQ,jcEko,VtokZ


In [7]:
register_function(
    "df_creator",
    lambda rows, cols, con: pd.DataFrame(np.random.rand(rows, cols)),
    generates_filepath=False,
)


fab_functions.extract_and_replace_functions("select * from mydfcreator(1,2)")

2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Extracted subquery: mydfcreator.None at 26:29
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Replacing mydfcreator(1,2) with __fabduck_1


('select * from __fabduck_1',
 {'__fabduck_1': ContextObject(functionname='mydfcreator', functioncall='mydfcreator(1,2)', params='1,2', data=None, name='__fabduck_1', is_file=False)})

In [8]:
register_function(
    "dfcreate",
    lambda rows, cols, filename, con: pd.DataFrame(
        np.random.rand(rows, cols)
    ).to_parquet(filename),
    generates_filepath=True,
)

duckdb.connect().execute("select * from dfcreate(3,4)").df()

2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    # Statements: 1, ['select * from dfcreate(3,4)']
2025-08-28 18:31:28 fabduckdb.fab_execute DEBUG    Processing: select * from dfcreate(3,4)
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Extracted subquery: dfcreate.None at 23:26
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Replacing dfcreate(3,4) with __fabduck_1.parquet
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    select * from dfcreate(3,4) rewritten to select * from __fabduck_1.parquet, {'__fabduck_1.parquet': ContextObject(functionname='dfcreate', functioncall='dfcreate(3,4)', params="3,4,filename='__fabduck_1.parquet'", data=None, name='__fabduck_1', is_file=True)}
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Executing subquery dfcreate(3,4)
2025-08-28 18:31:28 fabduckdb.table_functions.fab_functions DEBUG    Running dfcreate against 3,4,filename='__fabduck_1.parquet'
2025-08-28 18:31:28

,0,1,2,3
0,0.204270,0.912623,0.627889,0.485056
1,0.015628,0.120711,0.884183,0.548982
2,0.943425,0.480554,0.470649,0.994520
